In [2]:
import numpy as np

rng = np.random.default_rng()

Субъект (особь) - набор индексов разработчиков, назначенных на каждую задачу по порядку. (по факту: особь - ответ)

In [3]:
CATEGORIES_COUNT = 4

In [4]:
with open("input.txt") as f:
    tasks_count = N = np.loadtxt(f, max_rows=1, dtype=np.int64)
    tasks_categories = np.loadtxt(f, max_rows=1, dtype=np.int64) - 1  # (n,)
    tasks_times = np.loadtxt(f, max_rows=1, dtype=np.float64)  # (n,)
    developers_count = M = np.loadtxt(f, max_rows=1, dtype=np.int64)
    developers_coefficients = np.loadtxt(f, max_rows=M, dtype=np.float64)  # (n, m)

In [5]:
def create_random_population(genes_count: int, subjects_count: int) -> np.ndarray:
    return rng.integers(0, developers_count, size=(subjects_count, genes_count))

In [274]:
I_categories = np.eye(CATEGORIES_COUNT, dtype=bool)

def get_fitness_minimal(population: np.ndarray) -> np.ndarray:
    assert population.ndim == 2

    times = developers_coefficients[population][..., I_categories[tasks_categories]]
    times *= tasks_times

    mask = population == np.arange(developers_count)[:, None, None]
    masked = np.where(mask, times, 0)
    return masked.sum(axis=-1).max(axis=0)

def get_fitness_answer(population: np.ndarray) -> np.ndarray:
    fit = get_fitness_minimal(population)
    return 100 / (1e-10 + fit)

def get_fitness(population: np.ndarray) -> np.ndarray:
    fit = get_fitness_minimal(population)
    return np.exp(-fit/10)

In [159]:
def su_sampling(fitness: np.ndarray, n: int, start: float = None) -> np.ndarray:
    _fitness = fitness.copy()
    _fitness.sort()
    _fitness = _fitness[::-1]
    argsort = fitness.argsort()[::-1]

    fitness_cumsum = np.cumsum(_fitness)
    h = fitness_cumsum[-1] / n

    if start is None:
        start = rng.uniform(0, h)

    pointers = np.arange(start, start + (n - 0.5) * h, h)
    mask = pointers[:, None] < fitness_cumsum
    return argsort[mask.argmax(axis=-1)]

In [160]:
def one_point_crossover(population_l: np.ndarray, population_r: np.ndarray, point: np.ndarray) -> np.ndarray:
    mask = np.arange(population_l.shape[1]) < point[:, None]
    return np.where(mask, population_l, population_r)

In [161]:
def create_new_population(old_population: np.ndarray, sampled_idx: np.ndarray, count: int) -> np.ndarray:
    sampled_population = old_population[sampled_idx]

    subject1_idx = rng.choice(sampled_population.shape[0], count)
    subject2_idx = (subject1_idx + rng.integers(1, sampled_population.shape[0], count)) % sampled_population.shape[0]

    points = rng.choice(sampled_population.shape[1], count)

    new_population = one_point_crossover(sampled_population[subject1_idx], sampled_population[subject2_idx], points)

    return new_population

In [162]:
def create_mutations(population: np.ndarray, gen_mutation_chance: float = 0.01) -> np.ndarray:
    gen_mutation_chance = gen_mutation_chance or 0.01
    new_population = population.copy()

    mutated_mask = rng.random(new_population.shape) < gen_mutation_chance
    total_mutated = mutated_mask.sum()

    new_population[mutated_mask] += rng.integers(1, developers_count, total_mutated)
    new_population[mutated_mask] %= developers_count
    return new_population

### !

In [34]:
POPULATION_COUNT = 1000

In [243]:
pop = create_random_population(tasks_count, POPULATION_COUNT)
pop

array([[4, 6, 8, ..., 5, 6, 2],
       [0, 7, 8, ..., 2, 6, 8],
       [7, 1, 4, ..., 9, 7, 9],
       ...,
       [2, 2, 1, ..., 5, 0, 7],
       [8, 8, 6, ..., 0, 6, 9],
       [4, 8, 3, ..., 0, 9, 2]], shape=(1000, 1000))

In [276]:
def genetic_iterations(start_population: np.ndarray, iterations_number: int, mutations_chance: float = None) -> np.ndarray:
    _population = start_population.copy()
    for _ in range(iterations_number):
        fitness = get_fitness(_population)
        _f_max = fitness.max()
        # print(_f_max, end=" ")
        sampled_subjects_idx = su_sampling(fitness, min(POPULATION_COUNT//20, 10))
        new_population = create_new_population(_population, sampled_subjects_idx, POPULATION_COUNT)
        _population = create_mutations(new_population, mutations_chance)
    # print()
    return _population

In [46]:
# with open(f"output_info.txt", "w") as fi:
#     print("", end="", file=fi)

In [278]:
def save_data(_population: np.ndarray):
    _fit = get_fitness(_population)
    _fit_ans = get_fitness_answer(_population)
    _fit_min = get_fitness_minimal(_population)

    _arg = _fit.argmax()
    _max = _fit.max()
    _ans = _fit_ans.max()
    _min = _fit_min.min()
    print(_ans, _max, _min, _arg)
    with open(f"output_info.txt", "a") as fi:
        print(_ans, _max, _min, _arg, file=fi)
        print(*(_population[_arg] + 1), file=fi)

In [165]:
save_data(pop)

0.014688342743664703 0 6808.12 0


In [213]:
from io import StringIO
# found_min = "1 " * int(tasks_count)
found_min = ""
found_min = np.loadtxt(StringIO(found_min), dtype=int)
# print(*found_min)
# found_min

In [214]:
pop = found_min[None].repeat(POPULATION_COUNT, axis=0) - 1

In [ ]:
for _j in range(10):
    pop = genetic_iterations(pop, 50, mutations_chance=0.001)
    save_data(pop)

In [216]:
max_arg = get_fitness(pop).argsort()[-10:]
pop = pop[max_arg].repeat(POPULATION_COUNT//10, axis=0)

In [249]:
pop[1:] = create_mutations(pop[1:])

In [279]:
for _j in range(100):
    pop = genetic_iterations(pop, 50, mutations_chance=0.001)
    save_data(pop)

0.16143874206929573 1.254575804216861e-27 619.43 432
0.16333197223353804 2.5722924645079905e-27 612.25 598
0.16267833612595164 2.0113326470071512e-27 614.7099999999999 374
0.1629208449074751 2.204052058099814e-27 613.7950000000001 570
0.16306164545503768 2.3240178294467997e-27 613.265 907
0.16218758616212886 1.6699563875351424e-27 616.5699999999999 643
0.1626571674880589 1.9953061771847786e-27 614.79 421
0.16243918682940434 1.8373007499780744e-27 615.615 820
0.16320135783527054 2.449288344770171e-27 612.74 433
0.16289563276805893 2.1832127071937443e-27 613.89 234
0.16247745625291848 1.8641356940281815e-27 615.47 24
0.16369157233937132 2.9426116720153225e-27 610.905 345
0.16285318785112565 2.148559270641227e-27 614.05 196
0.16236533824756452 1.786569875593888e-27 615.895 906
0.16310286897943874 2.3603207264436007e-27 613.11 162
0.16330529925693424 2.546697726840397e-27 612.3499999999999 656
0.16364067779966066 2.8872298436762153e-27 611.095 278
0.16279486219412265 2.1018071259335188e-27